## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-14-12-47-03 +0000,startribune,Thousands of flights canceled as airlines work...,https://www.startribune.com/thousands-of-fligh...
1,2026-11-14-12-46-20 +0000,bbc,Welsh Labour leader says party focused too muc...,https://www.bbc.co.uk/news/articles/c6jre3q9jj...
2,2026-11-14-12-39-41 +0000,bbc,Young people should have legal guarantee of jo...,https://www.bbc.co.uk/news/articles/c69vwp09d7...
3,2026-11-14-12-35-33 +0000,cbc,Carney says Canada will only sign a U.S. trade...,https://www.cbc.ca/news/politics/canada-us-tra...
4,2026-11-14-12-28-19 +0000,nypost,Beloved doctor dead at 53 after freak accident...,https://nypost.com/2026/11/14/us-news/beloved-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2451/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
240,trump,33
397,nvidia,26
7,ground,22
127,new,22
118,huang,19


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
115,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...,136
337,2026-11-13-16-01-00 +0000,wsj,Treasury Secretary Scott Bessent said Howard L...,https://www.wsj.com/politics/policy/bessent-sa...,118
121,2026-11-14-02-00-00 +0000,wsj,The White House ordered officials this summer ...,https://www.wsj.com/politics/policy/china-vape...,98
140,2026-11-14-00-49-44 +0000,latimes,Democrats press Trump administration for answe...,https://www.latimes.com/politics/story/2026-11...,96
165,2026-11-13-23-12-22 +0000,nypost,Jensen Huang commemorative plaque gone from fa...,https://nypost.com/2026/11/13/business/jensen-...,94


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
115,136,2026-11-14-02-07-00 +0000,wsj,"More than almost any CEO, Jensen Huang helped ...",https://www.wsj.com/politics/policy/nvidia-jen...
354,84,2026-11-13-14-55-01 +0000,nypost,Treasury Secretary Scott Bessent points finger...,https://nypost.com/2026/11/13/us-news/scott-be...
156,84,2026-11-13-23-52-22 +0000,latimes,"LAX, other airports filled with stranded passe...",https://www.latimes.com/california/story/2026-...
121,65,2026-11-14-02-00-00 +0000,wsj,The White House ordered officials this summer ...,https://www.wsj.com/politics/policy/china-vape...
249,62,2026-11-13-20-07-00 +0000,wsj,Rep. Lauren Boebert lost her deep-red Colorado...,https://www.wsj.com/politics/elections/lauren-...
240,62,2026-11-13-20-29-03 +0000,wapo,"Before he was Pope Leo, he was their bishop. N...",https://www.washingtonpost.com/world/2026/11/1...
225,53,2026-11-13-21-06-41 +0000,cbc,'Miracle on the Mississippi': All aboard river...,https://www.cbc.ca/news/world/tanker-hits-crui...
117,52,2026-11-14-02-03-00 +0000,wsj,California voters had a chance to impose the n...,https://www.wsj.com/politics/california-voters...
71,50,2026-11-14-06-42-27 +0000,bbc,Microsoft says services fully restored after d...,https://www.bbc.co.uk/news/articles/c6rrg51rkz...
226,48,2026-11-13-21-06-14 +0000,nypost,"Trump will have historic meeting with Xi, Puti...",https://nypost.com/2026/11/13/us-news/trump-wi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
